# Fine-Tuning a Small Language Model with LoRA — Astrologer Personalities

In this exercise, you'll fine-tune **Qwen2.5-0.5B-Instruct** into TWO different astrologer personas using LoRA adapters:

| Persona | Style |
|---------|-------|
| **Mystic Maya** | Warm, poetic, spiritual. Uses metaphors and encouragement. |
| **Brutal Brad** | Blunt, sarcastic, funny. No sugarcoating, delivers truth bombs. |

Same base model, same questions, completely different personalities — just by swapping the LoRA adapter.

### What you'll learn
1. How to load a pretrained model and tokenizer
2. How to prepare personality-specific training data
3. How to apply **LoRA** (Low-Rank Adaptation) to fine-tune efficiently
4. How to train TWO adapters and swap between them

### Why LoRA?
Instead of updating all the model's parameters, LoRA freezes the original weights and injects small trainable "adapter" layers. This means:
- Much less memory needed
- Much faster training
- The adapter is tiny (a few MB) compared to the full model (~1 GB)

---
## Step 0: Setup (Colab)

**Important:** Always open this notebook via **File > Open notebook > GitHub** tab in Colab, so you get the latest version.

**Before running:** Go to **Runtime > Change runtime type > T4 GPU**

Then run the cell below to clone the training data and install dependencies.

In [ ]:
# Clone the repo to get the training data files
!rm -rf /content/fine-tuning-exercise
!git clone https://github.com/arlocinsights/fine-tuning-exercise.git /content/fine-tuning-exercise
%cd /content/fine-tuning-exercise

# Install dependencies
!pip install -q torch transformers datasets peft trl accelerate bitsandbytes

In [ ]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

---
## Step 1: Load the Model and Tokenizer

We'll use `Qwen2.5-0.5B-Instruct` — small enough to fine-tune on a free Colab T4 GPU.

**TODO:** Fill in the model name.

In [ ]:
# TODO: Fill in the model name (hint: Qwen/Qwen2.5-0.5B-Instruct)
MODEL_NAME = "___"

print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"Model loaded! Parameters: {base_model.num_parameters():,}")

---
## Step 2: Test the Base Model (Before Fine-Tuning)

Let's see how the base model answers astrology questions. Notice how generic and bland the responses are — no personality at all.

In [ ]:
def ask_model(model, tokenizer, question):
    """Send a question to the model and print the response."""
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {question}")
    print(f"A: {response}")
    print("-" * 50)

In [ ]:
test_questions = [
    "What's an Aries like in a relationship?",
    "I'm a Scorpio — what career suits me?",
    "Are Gemini and Virgo compatible?",
    "I keep dating Leos and it never works out. Why?",
]

print("=" * 50)
print("BASE MODEL RESPONSES (generic, no personality)")
print("=" * 50)
for q in test_questions:
    ask_model(base_model, tokenizer, q)

---
## Step 3: Prepare Training Data — Two Personalities

We have two JSON files with training data, one per persona. Each contains Q&A pairs where the answers are written in that persona's voice.

- `training_data/maya/maya_training.json` — warm, poetic, spiritual
- `training_data/brad/brad_training.json` — blunt, sarcastic, funny

**TODO:** Open each file (in the Colab file browser on the left) and add 3 more examples. Keep the personality consistent!

In [ ]:
REPO_DIR = "/content/fine-tuning-exercise"

with open(f"{REPO_DIR}/training_data/maya/maya_training.json") as f:
    maya_examples = json.load(f)

with open(f"{REPO_DIR}/training_data/brad/brad_training.json") as f:
    brad_examples = json.load(f)

print(f"Mystic Maya examples: {len(maya_examples)}")
print(f"Brutal Brad examples: {len(brad_examples)}")

# Preview one example from each
print("\n--- Maya sample ---")
print(f"Q: {maya_examples[0]['question']}")
print(f"A: {maya_examples[0]['answer'][:150]}...")

print("\n--- Brad sample ---")
print(f"Q: {brad_examples[0]['question']}")
print(f"A: {brad_examples[0]['answer'][:150]}...")

---
## Step 4: Format Data for Training

Each persona gets a different **system prompt** to reinforce the personality. We convert the Q&A pairs into the chat format the model expects.

In [ ]:
MAYA_SYSTEM = "You are Mystic Maya, a warm and poetic astrologer. You speak with spiritual metaphors, gentle encouragement, and cosmic wisdom. Your readings feel like a warm hug from the universe."
BRAD_SYSTEM = "You are Brutal Brad, a blunt and sarcastic astrologer. You deliver honest readings with sharp humor and zero sugarcoating. You're funny but always accurate."

def format_examples(examples, system_prompt):
    """Convert Q&A pairs into chat template format with a persona system prompt."""
    formatted = []
    for ex in examples:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": ex["question"]},
            {"role": "assistant", "content": ex["answer"]},
        ]
        formatted.append({"text": tokenizer.apply_chat_template(messages, tokenize=False)})
    return Dataset.from_list(formatted)

maya_dataset = format_examples(maya_examples, MAYA_SYSTEM)
brad_dataset = format_examples(brad_examples, BRAD_SYSTEM)

print("Formatted Maya example:")
print(maya_dataset[0]["text"][:300] + "...")

---
## Step 5: Configure LoRA

Key parameters:
- **r** (rank): Size of the low-rank matrices. Smaller = fewer parameters to train.
- **lora_alpha**: Scaling factor. Usually set to 2x the rank.
- **target_modules**: Which layers of the model to apply LoRA to.

**TODO:** Set the rank. Start with `8`, then later try `4` or `16` to see the difference.

In [ ]:
lora_config = LoraConfig(
    r=___,                                   # TODO: Set the rank (try 8)
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

**Question:** What percentage of the model's total parameters will be trainable with LoRA? (You'll see the answer when training starts.)

---
## Step 6: Train Mystic Maya

**TODO:** Set the number of training epochs. Start with `3`.

In [ ]:
print("=" * 50)
print("TRAINING MYSTIC MAYA")
print("=" * 50)

maya_model = get_peft_model(base_model, lora_config)
maya_model.print_trainable_parameters()

maya_training_args = TrainingArguments(
    output_dir="./maya-adapter",
    num_train_epochs=___,                    # TODO: Set number of epochs (try 3)
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
)

maya_trainer = SFTTrainer(
    model=maya_model,
    args=maya_training_args,
    train_dataset=maya_dataset,
)

maya_trainer.train()
maya_model.save_pretrained("./maya-adapter/final")
print("Mystic Maya training complete!")

---
## Step 7: Train Brutal Brad

We reload the base model fresh and train a completely separate adapter.

In [ ]:
print("=" * 50)
print("TRAINING BRUTAL BRAD")
print("=" * 50)

# Reload base model fresh for the second adapter
base_model_2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

brad_model = get_peft_model(base_model_2, lora_config)
brad_model.print_trainable_parameters()

brad_training_args = TrainingArguments(
    output_dir="./brad-adapter",
    num_train_epochs=___,                    # TODO: Same number of epochs as Maya
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
)

brad_trainer = SFTTrainer(
    model=brad_model,
    args=brad_training_args,
    train_dataset=brad_dataset,
)

brad_trainer.train()
brad_model.save_pretrained("./brad-adapter/final")
print("Brutal Brad training complete!")

---
## Step 8: Compare All Three — Base vs Maya vs Brad

This is the payoff! Same questions, three completely different responses. We test on questions that were **NOT** in the training data to see if the personality generalizes.

In [ ]:
# Reload base model and attach adapters
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

maya = PeftModel.from_pretrained(base, "./maya-adapter/final")
brad = PeftModel.from_pretrained(base, "./brad-adapter/final")

In [ ]:
unseen_questions = [
    "What happens during Saturn return?",
    "I'm a Capricorn — will I find love this year?",
]

for q in unseen_questions:
    print(f"\n{'=' * 60}")
    print(f"QUESTION: {q}")
    print("=" * 60)

    print("\n[Base Model]")
    ask_model(base, tokenizer, q)

    print("[Mystic Maya]")
    ask_model(maya, tokenizer, q)

    print("[Brutal Brad]")
    ask_model(brad, tokenizer, q)

---
## Step 9: Check Adapter Sizes

Both personality adapters are tiny compared to the full model. This is the power of LoRA — you can store dozens of personalities cheaply.

In [ ]:
def get_adapter_size(path):
    return sum(
        os.path.getsize(os.path.join(path, f))
        for f in os.listdir(path)
        if f.endswith(".safetensors")
    )

maya_size = get_adapter_size("./maya-adapter/final")
brad_size = get_adapter_size("./brad-adapter/final")

print(f"Mystic Maya adapter: {maya_size / 1024 / 1024:.2f} MB")
print(f"Brutal Brad adapter: {brad_size / 1024 / 1024:.2f} MB")
print(f"Full base model:     ~1,000 MB")
print(f"\nTwo complete personalities stored in just {(maya_size + brad_size) / 1024 / 1024:.2f} MB!")

---
## Bonus Challenges

1. **Create a third persona** — a dramatic soap-opera astrologer, a Gen-Z astrologer who uses slang, or a skeptical scientist who reluctantly gives readings. Create a new JSON file in `training_data/` and add a training section above.

2. **Experiment with LoRA rank** — go back to Step 5 and try `r=4` vs `r=16` vs `r=32`. How does it affect how well the personality comes through?

3. **Add more data** — does going from 7 to 20+ examples per persona improve consistency on unseen questions?

4. **Try different target modules** — change `target_modules` to `["q_proj", "v_proj", "k_proj", "o_proj"]`. More modules = more trainable parameters. Does it help?

5. **Break it on purpose** — train for 50 epochs and see what overfitting looks like. Compare outputs to the 3-epoch version.

---

## Key Takeaways

- **Fine-tuning** adapts a pretrained model to behave differently — new style, new personality, new domain
- **LoRA** makes it practical by only training a tiny fraction of the model's parameters
- **Adapters are swappable** — one base model can serve multiple personalities at inference time
- This is how real apps offer different AI personas from a single model deployment